In [22]:
import os
import pandas as pd
import numpy as np
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from dotenv import load_dotenv
from pycoingecko import CoinGeckoAPI
import warnings

warnings.filterwarnings("ignore")

# Load environment variables
load_dotenv()



# Google Sheets configuration
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']
SERVICE_ACCOUNT_FILE = os.getenv('SERVICE_ACCOUNT_FILE')
SPREADSHEET_ID = '1156S3pm9vQ2JP5bE0yvpj_Y8FE55ecovHS_JvIys5dc'

# Authenticate and build the service
creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
service = build('sheets', 'v4', credentials=creds)

def get_allocation_data(sheet_name):
    """
    Pulls allocation data from the specified Google Sheets sheet.
    
    Parameters:
        sheet_name (str): The name of the sheet (e.g. "EXC", "HB", or "LC")
    
    Returns:
        pd.DataFrame: DataFrame built from the sheet values (header in first row)
                      with percentage strings converted to decimal values
    """
    range_name = f'{sheet_name}!A:AAZ'
    result = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=range_name
    ).execute()
    values = result.get('values', [])
    if not values:
        print(f'No data found in sheet {sheet_name}.')
        return None
    # Assume the first row is the header (tickers and possibly a "date" column)
    df = pd.DataFrame(values[1:], columns=values[0])
    
    # Convert percentage strings to decimal values
    for col in df.columns:
        if col.lower() not in ['date', 'data']:  # Skip date/data columns
            df[col] = df[col].apply(lambda x: float(x.replace('%', ''))/100 if isinstance(x, str) and '%' in x else x)
    
    return df

def parse_allocation_row(df):
    """
    Converts the latest (last) row of the allocation DataFrame to a dictionary of ticker: allocation,
    and extracts the start date.
    """
    allocation_date = None
    allocations = df.iloc[-1]
    allocation_dict = {}
    
    # Look for date column with case-insensitive match (including Portuguese "Data")
    date_column = next((col for col in df.columns if col.lower() in ['date', 'data']), None)
    if date_column:
        try:
            allocation_date = pd.to_datetime(allocations[date_column])
            print(f"Found date: {allocation_date}")
        except Exception as e:
            print(f"Error parsing date from column {date_column}:", e)
    else:
        print("No date column found in the sheet (looked for 'date' or 'data')")
    
    # Process allocations
    for ticker, value in allocations.items():
        if ticker.lower() not in ['date', 'data']:  # Skip date column
            if isinstance(value, str):
                value = value.strip()
                if value.endswith('%'):
                    try:
                        allocation = float(value.replace('%', '')) / 100
                    except ValueError:
                        allocation = 0.0
                else:
                    try:
                        allocation = float(value)
                    except ValueError:
                        allocation = 0.0
            else:
                try:
                    allocation = float(value)
                except Exception:
                    allocation = 0.0
            allocation_dict[ticker.upper()] = allocation
    
    return allocation_dict, allocation_date

def get_latest_allocations():
    """
    Pulls the latest allocation data for the three portfolios.
    The portfolios are defined by sheet names:
      carteira_EXC -> sheet "EXC"
      carteira_HB  -> sheet "HB"
      carteira_LC  -> sheet "LC"
    
    Returns:
         dict: Dictionary with keys 'carteira_EXC', 'carteira_HB', and 'carteira_LC'
               and each value is a tuple (allocation_dict, allocation_date).
    """
    portfolios = {
        'carteira_EXC': 'EXC',
        'carteira_HB': 'HB',
        'carteira_LC': 'LC'
    }
    all_allocations = {}
    for portfolio_key, sheet_name in portfolios.items():
        df = get_allocation_data(sheet_name)
        if df is not None:
            allocation_dict, allocation_date = parse_allocation_row(df)
            all_allocations[portfolio_key] = (allocation_dict, allocation_date)
    return all_allocations

def get_ticker_mapping(portfolio_assets):
    """
    Uses the CoinGecko API to retrieve the coins list and creates a mapping 
    from coin ticker (upper-case) to coin id.
    """
    cg = CoinGeckoAPI()
    coins_list = cg.get_coins_list()
    coins_df = pd.DataFrame(coins_list)
    
    # Define known mappings for ambiguous tickers
    known_mappings = {
        'VIRTUAL': 'virtual-protocol',
        'HYPE': 'hyperliquid',
        'YNE': 'yesnoerror'
    }
    
    # Create mapping by first checking known mappings, then filtering by portfolio_assets
    mapping = {}
    
    # Add known mappings first
    for ticker, coin_id in known_mappings.items():
        if coin_id in portfolio_assets:
            mapping[ticker] = coin_id
    
    # Add remaining mappings from CoinGecko
    filtered_coins_df = coins_df[coins_df['id'].isin(portfolio_assets)]
    for _, row in filtered_coins_df.iterrows():
        ticker = row['symbol'].upper()
        if ticker not in mapping:  # Don't override known mappings
            mapping[ticker] = row['id']
    
    return mapping

def build_dataframe(portfolio_assets, allocation_dict=None):
    """
    Builds a price close DataFrame using either all assets in portfolio_assets list
    or only the tickers with non-zero allocation if allocation_dict is provided.
    """
    mapping = get_ticker_mapping(portfolio_assets)
    price_data = {}
    
    if allocation_dict is not None:
        assets_to_process = {ticker: alloc for ticker, alloc in allocation_dict.items() if alloc > 0}
    else:
        reverse_mapping = {v: k for k, v in mapping.items()}
        assets_to_process = {reverse_mapping.get(asset_id, asset_id): 1 for asset_id in portfolio_assets}
    
    # Process each asset
    for ticker in assets_to_process:
        if ticker in mapping:
            coin_id = mapping[ticker]
            file_path = os.path.join("data/micro", "assetData", f"{coin_id}.csv")
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                if 'date' in df.columns and 'close' in df.columns:
                    df['date'] = pd.to_datetime(df['date'])
                    df = df.drop_duplicates(subset=['date'])  # Remove duplicate dates
                    df.set_index('date', inplace=True)
                    price_data[ticker] = df['close']
                else:
                    print(f"Required columns not found in {file_path}")
            else:
                print(f"File not found: {file_path}")
        else:
            print(f"Ticker {ticker} not found in ticker mapping.")
    
    if not price_data:
        return pd.DataFrame()
    
    # Create DataFrame with all series and ensure index is unique
    price_df = pd.DataFrame(price_data)
    price_df = price_df[~price_df.index.duplicated(keep='first')]  # Keep first occurrence of duplicate indices
    
    # Sort index to ensure chronological order
    price_df.sort_index(inplace=True)
    
    # For each column (asset), fill NaN values with the first available price
    for column in price_df.columns:
        first_valid_price = price_df[column].first_valid_index()
        if first_valid_price is not None:
            price_df[column].fillna(price_df[column][first_valid_price], inplace=True)
    
    return price_df

if __name__ == "__main__":
    # Import the portfolio lists from assetsRoster (only these three are used)
    from scripts.assetsRoster import carteira_EXC, carteira_HB, carteira_LC, carteira_AC

    # (1) Pull allocation data from Google Sheets and parse the percentages and allocation date.
    latest_allocations = get_latest_allocations()
    
    exc_alloc, exc_date = latest_allocations.get('carteira_EXC', ({}, None))
    hb_alloc, hb_date = latest_allocations.get('carteira_HB', ({}, None))
    lc_alloc, lc_date = latest_allocations.get('carteira_LC', ({}, None))
    ac_alloc, ac_date = latest_allocations.get('carteira_AC', ({}, None))
    # (3) Build a price close DataFrame for each portfolio using only non-zero allocations.
    # Each portfolio's price data will start on the allocation date extracted from the sheet.
    exc_price_df = build_dataframe(carteira_EXC, exc_alloc)
    hb_price_df = build_dataframe(carteira_HB, hb_alloc)
    lc_price_df = build_dataframe(carteira_LC, lc_alloc)
    ac_price_df = build_dataframe(carteira_AC, ac_alloc)
    # Print (or further process) the resulting DataFrames
    print("EXC Portfolio Price Data:")
    print(exc_price_df.head())
    
    print("\nHB Portfolio Price Data:")
    print(hb_price_df.head())
    
    print("\nLC Portfolio Price Data:")
    print(lc_price_df.head())

    print("\nLC Portfolio Price Data:")
    print(ac_price_df.head())

Found date: 2025-05-08 00:00:00
Found date: 2025-05-08 00:00:00
Found date: 2025-05-08 00:00:00
Ticker SUI not found in ticker mapping.
Ticker FET not found in ticker mapping.
Ticker DEEP not found in ticker mapping.
Ticker RAY not found in ticker mapping.
Ticker SYRUP not found in ticker mapping.
Ticker EUL not found in ticker mapping.
Ticker KMNO not found in ticker mapping.
EXC Portfolio Price Data:
               BTC       SOL       HNT        CRV    RENDER    MORPHO  \
date                                                                    
2013-04-27  135.30  0.957606  0.129365  15.372108  0.051188  1.268028   
2013-04-28  141.96  0.957606  0.129365  15.372108  0.051188  1.268028   
2013-04-29  135.30  0.957606  0.129365  15.372108  0.051188  1.268028   
2013-04-30  117.00  0.957606  0.129365  15.372108  0.051188  1.268028   
2013-05-01  103.43  0.957606  0.129365  15.372108  0.051188  1.268028   

             VIRTUAL      HYPE         TAO  
date                                 

In [49]:
cg = CoinGeckoAPI(os.getenv('GECKO_API_KEY'))
coin_data = cg.get_coin_by_id('ethervista')

In [53]:
def get_exchanges_by_volume(asset_id):
    """
    Retrieves a dataframe of available exchanges for a given asset, ordered by volume.
    
    Parameters:
        asset_id (str): The CoinGecko asset ID (e.g., 'bitcoin', 'ethereum', 'render-token')
        
    Returns:
        pd.DataFrame: DataFrame containing exchange information ordered by volume
    """
    try:
        # Get coin data from CoinGecko
        coin_data = cg.get_coin_by_id(asset_id)
        
        # Extract ticker data
        tickers = coin_data.get('tickers', [])
        
        if not tickers:
            return pd.DataFrame()
        
        # Create a list to store exchange data
        exchange_data = []
        
        for ticker in tickers:
            exchange_data.append({
                'Exchange': ticker.get('market', {}).get('name', 'Unknown'),
                'Pair': f"{ticker.get('base', '')}/{ticker.get('target', '')}",
                'Price': ticker.get('last', 0),
                'Volume (USD)': ticker.get('converted_volume', {}).get('usd', 0),
                'Trust Score': ticker.get('trust_score', 'unknown'),
                'Trading URL': ticker.get('trade_url', '')
            })
        
        # Convert to DataFrame and sort by volume
        exchange_df = pd.DataFrame(exchange_data)
        exchange_df = exchange_df.sort_values(by='Volume (USD)', ascending=False)
        
        return exchange_df
    
    except Exception as e:
        print(f"Error retrieving exchange data for {asset_id}: {e}")
        return pd.DataFrame()

# Example usage
render_exchanges = get_exchanges_by_volume('ethereum')
display(render_exchanges.head(10))

,Exchange,Pair,Price,Volume (USD),Trust Score,Trading URL
0,Binance,ETH/USDC,2350.21,3087001454,green,https://www.binance.com/en/trade/ETH_USDC?ref=...
1,Bitget,ETH/USDT,2348.50,1716257064,green,https://www.bitget.com/spot/ETHUSDT
2,OKX,ETH/USDT,2347.99,1700937614,green,https://www.okx.com/trade-spot/eth-usdt
11,BitMart,ETH/USDT,2345.41,1565661540,green,https://www.bitmart.com/trade/en?symbol=ETH_USDT
3,Crypto.com Exchange,ETH/USD,2350.30,1444026591,green,https://crypto.com/exchange/trade/spot/ETH_USD
5,Binance,ETH/FDUSD,2349.68,1409048212,green,https://www.binance.com/en/trade/ETH_FDUSD?ref...
4,Gate.io,ETH/USDT,2350.37,1379479153,green,https://www.gate.io/trade/ETH_USDT
10,DigiFinex,ETH/USDT,2348.62,1324617758,green,https://www.digifinex.com/en-ww/trade/USDT/ETH
6,Crypto.com Exchange,ETH/USDT,2352.30,1135887504,green,https://crypto.com/exchange/trade/spot/ETH_USDT
7,Bybit,ETH/USDT,2349.10,1074855288,green,https://www.bybit.com/trade/spot/ETH/USDT


In [37]:
coin_data.get('tickers')[2]

{'base': 'BTC',
 'target': 'USDT',
 'market': {'name': 'Bybit',
  'identifier': 'bybit_spot',
  'has_trading_incentive': False},
 'last': 103114.5,
 'volume': 22299.259509,
 'converted_last': {'btc': 1.000499, 'eth': 44.080424, 'usd': 103102},
 'converted_volume': {'btc': 22102, 'eth': 973766, 'usd': 2277589746},
 'trust_score': 'green',
 'bid_ask_spread_percentage': 0.010097,
 'timestamp': '2025-05-09T12:49:15+00:00',
 'last_traded_at': '2025-05-09T12:49:15+00:00',
 'last_fetch_at': '2025-05-09T12:51:13+00:00',
 'is_anomaly': False,
 'is_stale': False,
 'trade_url': 'https://www.bybit.com/trade/spot/BTC/USDT',
 'token_info_url': None,
 'coin_id': 'bitcoin',
 'target_coin_id': 'tether'}

In [21]:
get_allocation_data('AC')

,Data,BTC,ETH,SOL,MKR,LINK,RUNE,STX,IMX,UNI,...,VISTA,MORPHO,ENA,VIRTUAL,ONDO,YNE,USDT,TAO,CRV,RENDER
0,2024-07-30,0.50,0.15,0.15,0.03,0.03,0.03,0.03,0.03,0.02,...,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00
1,2024-07-31,0.50,0.15,0.15,0.03,0.03,0.03,0.03,0.03,0.02,...,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00
2,2024-08-01,0.50,0.15,0.15,0.03,0.03,0.03,0.03,0.03,0.02,...,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00
3,2024-08-02,0.50,0.15,0.15,0.03,0.03,0.03,0.03,0.03,0.02,...,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00
4,2024-08-03,0.50,0.15,0.15,0.03,0.03,0.03,0.03,0.03,0.02,...,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263,2025-04-19,0.50,0.05,0.05,0.00,0.05,0.00,0.00,0.00,0.02,...,0.02,0.02,0.03,0.02,0.05,0.02,0.1,0.00,0.00,0.00
264,2025-04-20,0.50,0.05,0.05,0.00,0.05,0.00,0.00,0.00,0.02,...,0.02,0.02,0.03,0.02,0.05,0.02,0.1,0.00,0.00,0.00
265,2025-04-21,0.50,0.05,0.05,0.00,0.05,0.00,0.00,0.00,0.02,...,0.02,0.02,0.03,0.02,0.05,0.02,0.1,0.00,0.00,0.00
266,2025-04-22,0.50,0.05,0.05,0.00,0.05,0.00,0.00,0.00,0.02,...,0.02,0.02,0.03,0.02,0.05,0.02,0.1,0.00,0.00,0.00


In [2]:
import pandas_ta as ta
import plotly.express as px

def analyze_portfolio_technicals(price_df, start_date=None):
    """
    Calculates current drawdown, max historical drawdown, and RSI for each asset
    and creates a scatter plot of RSI vs Drawdown.
    
    Parameters:
        price_df (pd.DataFrame): DataFrame with price history for portfolio assets
        start_date (str or datetime, optional): Start date for calculations
    """
    # Filter DataFrame by start_date if provided
    if start_date:
        price_df = price_df[price_df.index >= pd.to_datetime(start_date)]
    
    # Initialize DataFrames to store results
    rsi_df = pd.DataFrame()
    drawdown_df = pd.DataFrame()
    max_drawdowns = {}
    
    # Calculate RSI and Drawdown for each asset
    for column in price_df.columns:
        # Calculate 14-day RSI
        rsi_df[column] = ta.rsi(price_df[column], length=14)
        
        # Calculate Drawdown from start_date
        rolling_max = price_df[column].expanding().max()
        drawdown = (price_df[column] - rolling_max) / rolling_max * 100
        drawdown_df[column] = drawdown
        
        # Calculate max historical drawdown from start_date
        max_drawdowns[column] = drawdown.min()
    
    # Get current values
    current_rsi = rsi_df.iloc[-1]
    current_drawdown = drawdown_df.iloc[-1]
    
    # Create DataFrame for plotting
    plot_df = pd.DataFrame({
        'Asset': current_rsi.index,
        'RSI': current_rsi.values,
        'Drawdown': current_drawdown.values,
        'Max_Drawdown': [max_drawdowns[asset] for asset in current_rsi.index]
    })
    
    # Create scatter plot with Drawdown on Y-axis
    fig = px.scatter(
        plot_df,
        x='RSI',
        y='Drawdown',
        text='Asset',
        title=f'Portfolio Assets: Drawdown vs RSI (from {price_df.index[0].strftime("%Y-%m-%d")})',
        labels={
            'RSI': '14-day RSI',
            'Drawdown': 'Current Drawdown (%)'
        }
    )
    
        # Update layout
    fig.update_traces(
        textposition='top center',
        marker=dict(size=8),
        textfont=dict(size=8)  # Add this line to make asset names smaller
    )
    fig.update_layout(
        plot_bgcolor='white',
        showlegend=False,
        hovermode='closest',
        xaxis=dict(
            range=[0, 100],
            gridcolor='lightgrey',
            dtick=10,
            tick0=0,
            tickmode='linear'
        ),
        yaxis=dict(
            gridcolor='lightgrey'
        )
    )
    
    # Add RSI reference lines (vertical)
    fig.add_vline(x=30, line_dash="dash", line_color="green", opacity=0.5)
    fig.add_vline(x=70, line_dash="dash", line_color="red", opacity=0.5)
    
    # Show plot
    fig.show()
    
    return plot_df

# Usage example:
# 

In [ ]:
analyze_portfolio_technicals(roster_price_df, start_date='2023-01-01')


In [10]:
def analyze_asset_ma_distance_3d(price_df, start_date=None):
    """
    Calculates the relative distance of each asset's last price from its moving averages (7-day, 30-day, 90-day)
    and plots a 3D scatter plot where each axis represents one of these distances.
    
    The relative distance is defined as: (last_price - moving_average) / moving_average.
    
    The function also returns breadth measurements, i.e. the percentage of assets with a positive relative distance,
    for each moving average period.
    
    Parameters:
        price_df (pd.DataFrame): DataFrame with price history for portfolio assets.
                                 Columns represent assets; index should be datetime.
        start_date (str or datetime, optional): Only use data from this date onward.
    
    Returns:
        tuple: (fig, output) where fig is a Plotly 3D scatter plot figure and output is a dictionary with:
            - breadth_7d: Percentage of assets with relative distance > 0 for the 7-day MA.
            - breadth_30d: Percentage of assets with relative distance > 0 for the 30-day MA.
            - breadth_90d: Percentage of assets with relative distance > 0 for the 90-day MA.
    """
    import pandas as pd
    import numpy as np
    import plotly.express as px

    # Filter the DataFrame by start_date if provided
    if start_date:
        price_df = price_df[price_df.index >= pd.to_datetime(start_date)]
    
    assets = []
    dist_7d = []
    dist_30d = []
    dist_90d = []
    
    # For each asset, compute the last price and its moving averages using a rolling window.
    # We assume that there is enough data for a 90-day window.
    for asset in price_df.columns:
        series = price_df[asset].dropna()
        if len(series) < 90:  # Skip asset if not enough history
            continue
        
        last_price = series.iloc[-1]
        
        # Compute moving averages
        sma7 = series.rolling(window=90).mean().iloc[-1]
        sma30 = series.rolling(window=180).mean().iloc[-1]
        sma90 = series.rolling(window=365).mean().iloc[-1]
        
        # In case any of the moving averages are nan, skip the asset
        if pd.isna(sma7) or pd.isna(sma30) or pd.isna(sma90):
            continue
        
        # Calculate relative distance: (price - moving average) / moving average
        d7 = (last_price - sma7) / sma7
        d30 = (last_price - sma30) / sma30
        d90 = (last_price - sma90) / sma90
        
        assets.append(asset)
        dist_7d.append(d7)
        dist_30d.append(d30)
        dist_90d.append(d90)
    
    # Create a DataFrame with the calculated distances for plotting.
    distance_df = pd.DataFrame({
        'Asset': assets,
        'dist_7d': dist_7d,
        'dist_30d': dist_30d,
        'dist_90d': dist_90d
    })
    
    # Calculate the breadth for each time frame: the percentage of assets with a positive distance.
    if len(distance_df) > 0:
        breadth_7d = (distance_df['dist_7d'] > 0).mean() * 100
        breadth_30d = (distance_df['dist_30d'] > 0).mean() * 100
        breadth_90d = (distance_df['dist_90d'] > 0).mean() * 100
    else:
        breadth_7d = breadth_30d = breadth_90d = None
    
    output = {
        'breadth_7d': breadth_7d,
        'breadth_30d': breadth_30d,
        'breadth_90d': breadth_90d
    }
    
    # Create a 3D scatter plot of the distances using Plotly Express.
    fig = px.scatter_3d(
        distance_df,
        x='dist_7d',
        y='dist_30d',
        z='dist_90d',
        text='Asset',
        title='Asset Relative Distance from MAs (7d, 30d, 90d)',
        labels={
            'dist_7d': 'Distance from 7d MA',
            'dist_30d': 'Distance from 30d MA',
            'dist_90d': 'Distance from 90d MA'
        }
    )
    
    fig.update_traces(marker=dict(size=5), textposition='top center', textfont=dict(size=10))
    fig.update_layout(scene=dict(
        xaxis_title='Relative Distance from 7d MA',
        yaxis_title='Relative Distance from 30d MA',
        zaxis_title='Relative Distance from 90d MA'
    ))
    
    fig.show()
    
    return fig, output

In [12]:
# Assuming roster_price_df holds your historical price data
fig, breadth_measurements = analyze_asset_ma_distance_3d(exc_price_df, start_date='2023-01-01')
print(breadth_measurements)

{'breadth_7d': 60.0, 'breadth_30d': 30.0, 'breadth_90d': 30.0}


In [13]:
# Function to fetch and parse candle data from CoinGecko
def fetch_coin_candle_data(coin_id, days=30):
    """
    Fetch OHLC (candle) data for a specific coin from CoinGecko and parse it.
    
    Parameters:
    -----------
    coin_id : str
        The CoinGecko ID of the coin (e.g., 'bitcoin', 'ethereum')
    days : int
        Number of days of historical data to fetch
        
    Returns:
    --------
    pandas.DataFrame
        Parsed OHLC data with columns [date, open, high, low, close]
    """
    import os
    import time
    import pandas as pd
    from datetime import datetime, timedelta
    from pycoingecko import CoinGeckoAPI
    from dotenv import load_dotenv
    
    # Load environment variables
    load_dotenv()
    
    # Initialize the CoinGecko API
    cg = CoinGeckoAPI(api_key=os.getenv('GECKO_API_KEY'))
    
    try:
        # Calculate date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days)
        
        # Convert to timestamps
        from_timestamp = int(start_date.timestamp())
        to_timestamp = int(end_date.timestamp())
        
        print(f"🔄 Fetching {coin_id} candle data from {start_date.date()} to {end_date.date()}")
        
        # Fetch data from the API
        data = cg.get_coin_ohlc_by_id_range(
            id=coin_id,
            vs_currency='usd',
            from_timestamp=from_timestamp,
            to_timestamp=to_timestamp,
            interval='daily'
        )
        
        if data:
            # Parse the data
            df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close'])
            df['date'] = pd.to_datetime(df['timestamp'], unit='ms')
            df = df[['date', 'open', 'high', 'low', 'close']]
            print(f"✅ Fetched {len(df)} candle records")
            return df
        else:
            print(f"❌ No data returned for {coin_id}")
            return None
    
    except Exception as e:
        print(f"❌ Error fetching candle data for {coin_id}: {e}")
        return None

# Test the function with Bitcoin data
bitcoin_candles_df = fetch_coin_candle_data('bitcoin', days=30)
if bitcoin_candles_df is not None:
    print("\nBitcoin OHLC data:")
    print(bitcoin_candles_df.head())
    
    # Display basic statistics
    print("\nBasic statistics:")
    print(f"Date range: {bitcoin_candles_df['date'].min()} to {bitcoin_candles_df['date'].max()}")
    print(f"Average close price: ${bitcoin_candles_df['close'].mean():.2f}")
    print(f"Min close price: ${bitcoin_candles_df['close'].min():.2f}")
    print(f"Max close price: ${bitcoin_candles_df['close'].max():.2f}")
    
    # Plot the candle data
    import matplotlib.pyplot as plt
    import mplfinance as mpf
    
    # Convert to format required by mplfinance
    plot_df = bitcoin_candles_df.set_index('date')
    
    # Plot candlestick chart
    mpf.plot(plot_df, type='candle', style='yahoo', 
             title='Bitcoin Price (USD)', 
             ylabel='Price (USD)',
             figsize=(12, 6))


🔄 Fetching bitcoin candle data from 2025-03-25 to 2025-04-24
✅ Fetched 31 candle records

Bitcoin OHLC data:
        date     open     high      low    close
0 2025-03-25  86114.0  88714.0  85560.0  87328.0
1 2025-03-26  87483.0  88430.0  86358.0  87521.0
2 2025-03-27  87424.0  88268.0  85863.0  86961.0
3 2025-03-28  86914.0  87773.0  85867.0  87227.0
4 2025-03-29  87213.0  87500.0  83609.0  84359.0

Basic statistics:
Date range: 2025-03-25 00:00:00 to 2025-04-24 00:00:00
Average close price: $84325.35
Min close price: $76329.00
Max close price: $93605.00


ModuleNotFoundError: No module named 'mplfinance'

In [10]:
bitcoin_candles_df.tail(5)

,date,open,high,low,close
26,2025-03-16,83955.0,84663.0,83689.0,84392.0
27,2025-03-17,84345.0,84693.0,82061.0,82611.0
28,2025-03-18,82570.0,84584.0,82570.0,84075.0
29,2025-03-19,83977.0,83977.0,81208.0,82780.0
30,2025-03-20,82618.0,86958.0,82618.0,86815.0


In [6]:
import os
import pandas as pd
from tqdm import tqdm

def fix_candle_dates():
    """
    Shifts all dates in candle data files back by one day to correct CoinGecko API peculiarity.
    """
    candle_folder = "./data/micro/candleData"
    
    if not os.path.exists(candle_folder):
        print("❌ Candle data folder not found!")
        return
        
    # Get list of all candle files
    candle_files = [f for f in os.listdir(candle_folder) if f.endswith('_candles.csv')]
    
    if not candle_files:
        print("No candle files found to process.")
        return
        
    print(f"🔄 Found {len(candle_files)} candle files to process")
    
    for filename in tqdm(candle_files, desc="Shifting dates"):
        file_path = os.path.join(candle_folder, filename)
        
        try:
            # Read the file
            df = pd.read_csv(file_path)
            
            # Check if file has data and 'date' column
            if df.empty or 'date' not in df.columns:
                print(f"⚠️ Skipping {filename}: Empty or missing date column")
                continue
                
            # Convert date column to datetime
            df['date'] = pd.to_datetime(df['date'])
            
            # Shift dates back by one day
            df['date'] = df['date'] - pd.Timedelta(days=1)
            
            # Format date back to string (optional, depends on your preference)
            df['date'] = df['date'].dt.strftime('%Y-%m-%d')
            
            # Save the fixed file (you might want to back up the originals first)
            df.to_csv(file_path, index=False)
            
        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")
    
    print("✅ All candle data files have been processed!")

if __name__ == "__main__":
    fix_candle_dates()

🔄 Found 132 candle files to process


Shifting dates: 100%|██████████| 132/132 [00:01<00:00, 99.79it/s] 

✅ All candle data files have been processed!
